**Human Movement Intelligence | Presented by: Reza Saadatyar (2026)**

**03 | Full Development Audit and Classical Grouped Cross-Validation**

![HARTH Stage 03 consolidated workflow](../../images/03_consolidated_workflow.svg)

**Objective:** Audit the complete 29-participant HARTH Development cohort, examine eight-class activity coverage and GroupKFold feasibility, and compare Dummy, Logistic Regression and Random Forest over five participant-disjoint folds. Report paired held-out-participant differences between Random Forest and Logistic Regression.

**Dataset:** Official pinned NTNU AI Lab HARTH v2.0 recordings with six lower-back/thigh acceleration channels at a nominal 50 Hz. Timestamp-safe, label-pure 100-sample windows use a 50-sample stride and five deterministic statistics per channel (30 features). The supervised task is the eight documented source classes (1–8).

**Notebook scope:** Reuse the **ten-person Stage 02 feature CSV** from `output/`, acquire and process only the **19 remaining Development participants** once, and create one aligned 29-person feature table. The audit and classical evaluation then consume these same rows without rebuilding the full cohort.

**Evaluation boundary:** S023 and S032 remain excluded from every Development model fit and score. These subjects were already inspected in earlier exploratory work; later same-study assessment must not be called strictly blind or external. Within each grouped fold, all fitted preprocessing and models use Train participants only.

**Reading order:** saved Stage 02 features → remaining-source inventory and preparation → full Development cohort → source exclusions and activity support → five-fold classical CV → paired participant-level interpretation → current Stage 03 outputs.

**Outputs:** Stage-owned CSV and PNG files are saved under the `03_` prefix in `wearable_activity_recognition/output/`, including a reusable aligned 29-person feature table. This new notebook does not inherit historical approvals, hash comparisons or administrative Quality Gates. Numerical results require execution of the current notebook.

**00. Colab Repository Setup**

Use the established independent Colab checkout shared by the earlier HARTH stages.

In [ ]:
# ======================== 00. Colab Repository Setup =========================

# -------------------------- 1. Repository Checkout ---------------------------
from pathlib import Path
import subprocess

repo_url = "https://github.com/RezaSaadatyar/human-movement-intelligence.git"
repo_path = Path("/content") / repo_url.rsplit("/", 1)[-1].removesuffix(".git")

if not repo_path.exists():
    subprocess.run(["git", "clone", "-q", repo_url, str(repo_path)], check=True)

# ---------------------------- 2. Open Repository -----------------------------
%cd {repo_path}

**01. Imports and Research Environment**

The project setup, scientific imports and participant protocol are grouped explicitly. The output files use one current stage directory.

In [ ]:
# =================== 01. Imports and Research Environment ====================

# ------------------ 1. Repository and Notebook Environment -------------------
from pathlib import Path
from python_modules.core.colab_setup import setup_colab
root = setup_colab("wearable_activity_recognition")

# ------------------------ 2. Numerical Data Analysis -------------------------
import numpy as np
import pandas as pd

# ----------------- 3. Scientific Plots and Notebook Display ------------------
import matplotlib.pyplot as plt
from IPython.display import display
from core.plot_style import plot_style, save_figure
plot_style(font_family="DejaVu Sans", title_size=12.5,
           title_weight="normal", label_size=11.5, label_weight="normal",
           tick_size=11, legend_size=10, figure_dpi=100,
           figure_size=(9, 4))

# ------------- 4. HARTH Source and Grouped Evaluation Utilities --------------
from harth_data import (
    HARTH_SUBJECTS, ACTIVITY_NAMES, SENSOR_COLUMNS,
    prepare_harth_feature_cohort,
)
from core.data_splitting import audit_group_class_coverage
from core.group_cross_validation import (
    evaluate_group_cross_validation, compare_models_by_subject,
)
from core.output_download import download_outputs_from_colab

# ----------------- 5. Dataset and Stage Output Configuration -----------------
output_dir = root / "wearable_activity_recognition/output"
data_dir = root / "data/raw/harth_v2"
output_dir.mkdir(parents=True, exist_ok=True)
RESERVED_TEST = {"S023", "S032"}
DEVELOPMENT_SUBJECTS = sorted(set(HARTH_SUBJECTS) - RESERVED_TEST)
TEN_SUBJECTS = {
    "S024", "S025", "S027", "S028", "S034",
    "S035", "S029", "S030", "S031", "S033",
}
REMAINING_SUBJECTS = sorted(set(DEVELOPMENT_SUBJECTS) - TEN_SUBJECTS)
TASK_CLASSES = tuple(range(1, 9))
FEATURE_METHODS = ("mean", "std", "rms", "peak_to_peak", "waveform_length")
FEATURE_COLUMNS = [
    f"{c}__{m}" for c in SENSOR_COLUMNS for m in FEATURE_METHODS
]
WINDOW_COLUMNS = [
    "subject_id", "start_index", "end_index_exclusive",
    "label", "label_purity",
]
print("Development participants:", len(DEVELOPMENT_SUBJECTS))
print("Previously processed:", len(TEN_SUBJECTS))
print("Remaining original CSVs:", len(REMAINING_SUBJECTS))

**02. Read Saved Ten-Person Development Features**

Read the Stage 02 CSV rather than processing the same ten participant recordings again. Preserve source-index metadata, feature column order and previously audited excluded activity codes.

In [ ]:
# ============== 02. Read Current Stage 02 Development Features ===============

# ----------- 1. Load Previously Saved Source Windows and Features ------------
ten_saved = pd.read_csv(
    output_dir / "02_development10_statistical_features.csv"
)
ten_source = pd.read_csv(output_dir / "02_ten_dev_cohort_summary.csv")
ten_excluded = pd.read_csv(
    output_dir / "02_ten_dev_excluded_source_codes.csv"
)
window_fields = ["window_" + col for col in WINDOW_COLUMNS]
missing = set(window_fields + FEATURE_COLUMNS) - set(ten_saved.columns)
if missing:
    raise ValueError(f"Stage 02 columns unavailable: {sorted(missing)}")
ten_metadata = ten_saved[window_fields].copy()
ten_metadata.columns = WINDOW_COLUMNS
ten_features = ten_saved[FEATURE_COLUMNS].copy()

# ------------ 2. Maintain Source Identity and Aligned Task Labels ------------
if set(ten_metadata["subject_id"]) != TEN_SUBJECTS:
    raise ValueError("Stage 02 has unexpected participant IDs.")
if set(ten_metadata["subject_id"]) & RESERVED_TEST:
    raise ValueError("Reserved Test participants occur in Development.")
if not ten_metadata["label"].isin(TASK_CLASSES).all():
    raise ValueError("Non-task activity labels are present.")
if not np.isfinite(ten_features.to_numpy()).all():
    raise ValueError("Stage 02 features contain nonfinite values.")
display(ten_metadata.head())
print("Reused Stage 02 feature matrix:", ten_features.shape)

**03. Inspect and Process the Remaining Nineteen Participants**

Report local source availability, then load the remaining 19 original recordings only. The shared feature function performs the same time-continuous, label-pure windowing and five statistical transformations once per person, excluding out-of-task labels after windowing.

In [ ]:
# ============== 03. Prepare Remaining Original Source Features ===============

# ------------------ 1. Inspect the Current Source Inventory ------------------
inventory = pd.DataFrame([
    {"subject_id": subject,
     "source": ("Stage 02 saved features" if subject in TEN_SUBJECTS
                else "Official HARTH v2.0 CSV"),
     "original_csv_available": (data_dir / f"{subject}.csv").is_file()}
    for subject in DEVELOPMENT_SUBJECTS
])
display(inventory)
print("Remaining source files absent locally:", int((~inventory.loc[
    inventory["subject_id"].isin(REMAINING_SUBJECTS),
    "original_csv_available"
]).sum()))

# --------------- 2. Build Only the Newly Required Feature Sets ---------------
new_features, new_metadata, new_overview, new_excluded = (
    prepare_harth_feature_cohort(
        subjects=REMAINING_SUBJECTS, data_dir=data_dir,
        task_classes=TASK_CLASSES,
        window_size=100, step_size=50, sampling_rate=50,
    )
)
display(new_overview)
display(new_excluded)
print("New Development feature matrix:", new_features.shape)

**04. Create One Reusable Full Development Cohort**

Assemble the ten saved and nineteen newly processed feature groups in ascending source-participant order. Preserve alignment between 30 numerical features and the five original window metadata fields. Save the current 29-person Development table once.

In [ ]:
# ============= 04. Combine Twenty-Nine Participant Feature Sets ==============

# -------------- 1. Concatenate Features in Stable Subject Order --------------
feature_parts, metadata_parts = [], []
for subject in DEVELOPMENT_SUBJECTS:
    if subject in TEN_SUBJECTS:
        chosen_features, chosen_meta = ten_features, ten_metadata
    else:
        chosen_features, chosen_meta = new_features, new_metadata
    selected = chosen_meta["subject_id"].eq(subject)
    feature_parts.append(
        chosen_features.loc[selected].reset_index(drop=True)
    )
    metadata_parts.append(
        chosen_meta.loc[selected].reset_index(drop=True)
    )
features = pd.concat(feature_parts, ignore_index=True)
metadata = pd.concat(metadata_parts, ignore_index=True)

# -------------- 2. Check Essential Group and Feature Alignment ---------------
if len(features) != len(metadata):
    raise ValueError("Feature rows and labels are misaligned.")
if set(metadata["subject_id"]) != set(DEVELOPMENT_SUBJECTS):
    raise ValueError("Expected Development participants are missing.")
if set(metadata["subject_id"]) & RESERVED_TEST:
    raise ValueError("Reserved Test participants entered Development.")
if not np.isfinite(features.to_numpy()).all():
    raise ValueError("Development features contain nonfinite values.")

# -------------- 3. Save Current Full Development Feature Table ---------------
current_features = pd.concat([
    metadata.add_prefix("window_"), features
], axis=1)
full_file = output_dir / "03_development29_statistical_features.csv"
current_features.to_csv(full_file, index=False)
print("Full Development feature matrix:", features.shape)
print("Saved reusable table:", full_file)

**05. Summarise Source Coverage and Excluded Activity Codes**

Reuse the ten-person audit from Stage 02 and combine it with source information for the remaining 19 participants. Distinguish raw samples, valid signal windows and documented eight-class eligible windows; keep cycling and undocumented source-code exclusions visible.

In [ ]:
# ============ 05. Combine Source Coverage and Exclusion Summaries ============

# ---------- 1. Reconstruct the Existing Ten-Person Audit from CSVs -----------
ten_n = ten_metadata.groupby("subject_id").size()
ten_k = ten_metadata.groupby("subject_id")["label"].nunique()
ten_overview = ten_source[[
    "subject_id", "raw_samples", "retained_windows",
    "unmapped_raw_rows",
]].rename(columns={"unmapped_raw_rows": "undocumented_raw_rows"}).copy()
ten_overview["eligible_windows"] = (
    ten_overview["subject_id"].map(ten_n).astype(int)
)
ten_overview["excluded_windows"] = (
    ten_overview["retained_windows"] - ten_overview["eligible_windows"]
)
ten_overview["task_classes_present"] = (
    ten_overview["subject_id"].map(ten_k).astype(int)
)

# --------------- 2. Combine Source and Excluded-Label Records ----------------
overview = pd.concat([ten_overview, new_overview], ignore_index=True)
overview = overview.set_index("subject_id").loc[
    DEVELOPMENT_SUBJECTS
].reset_index()
excluded = pd.concat([ten_excluded, new_excluded], ignore_index=True)
excluded = excluded[[
    "subject_id", "source_code", "source_activity", "excluded_windows",
]].sort_values(["subject_id", "source_code"]).reset_index(drop=True)
display(overview)
display(excluded)
print("Total eight-class windows:", len(metadata))
print("Out-of-task retained source windows:",
      int(excluded["excluded_windows"].sum()))

**06. Audit the Five Participant-Disjoint Fold Splits**

Use the existing GroupKFold audit once on the complete cohort, preserving the per-participant activity matrix and fold-level class availability. Feasibility is not itself evidence of classification performance.

In [ ]:
# =============== 06. Audit Grouped Development Class Coverage ================

# ------------- 1. Audit Fold Membership and Participant Classes --------------
coverage, fold_audit = audit_group_class_coverage(
    labels=metadata["label"].to_numpy(),
    group_ids=metadata["subject_id"].to_numpy(),
    classes=TASK_CLASSES,
    n_splits=5,
)
coverage.columns = [f"{code}: {ACTIVITY_NAMES[code]}" for code in TASK_CLASSES]

# ------------------ 2. Inspect Fold and Class Availability -------------------
display(coverage)

display(fold_audit)
print("Missing training classes by fold:",
      fold_audit[["fold", "missing_train_classes"]].to_dict("records"))

print("Missing validation classes by fold:",
      fold_audit[["fold", "missing_validation_classes"]].to_dict("records"))

**07. Visualise Participant Activity and Class Support**

Produce the two-panel coverage plot from the original data audit: eligible windows by activity and participant, and the number of independent participants supporting each activity.

In [ ]:
# ============= 07. Plot Independent Participant Activity Support =============

# ---------------- 1. Display Source Activity Counts by Person ----------------
fig, axes = plt.subplots(2, 1, figsize=(14, 14))
matrix = coverage.to_numpy()
heat = axes[0].imshow(matrix, aspect="auto", cmap="YlGnBu")
axes[0].set_xticks(np.arange(coverage.shape[1]), coverage.columns,
                   rotation=40, ha="right")
axes[0].set_yticks(np.arange(coverage.shape[0]), coverage.index)
axes[0].set(title="HARTH v2.0: development subject/class coverage",
            xlabel="Documented activity", ylabel="Participant")
fig.colorbar(heat, ax=axes[0], label="Eligible windows")

subject_support = (coverage > 0).sum(axis=0)

axes[1].barh(subject_support.index, subject_support.to_numpy())
axes[1].set(title="Independent development participants per activity",
            xlabel="Number of participants", xlim=(0, len(DEVELOPMENT_SUBJECTS)))
axes[1].grid(axis="x", alpha=0.25)
fig.tight_layout()
save_figure(fig, output_dir / "03_development_class_coverage.png")
plt.show()

**08. Train Three Classical Models in Five Development Folds**

Fit the original Dummy, Logistic Regression and Random Forest baselines on five participant-disjoint folds. Each fold's Logistic Regression scaler and model are fitted exclusively on its Train windows.

In [ ]:
# =========== 08. Evaluate Three Development Models with GroupKFold ===========

# -------------- 1. Fit New Models within Each Development Fold ---------------
fold_results, subject_results, summary = evaluate_group_cross_validation(
    features=features.to_numpy(),
    labels=metadata["label"].to_numpy(),
    group_ids=metadata["subject_id"].to_numpy(),
    classes=TASK_CLASSES,
    n_splits=5, random_state=42,
)
display(summary.round(4))

# -------------------- 2. Display Fold-Level Model Metrics --------------------
display(fold_results[[
    "fold", "model", "validation_subjects",
    "validation_windows", "validation_classes_present",
    "macro_f1", "balanced_accuracy",
]].round(4))

**09. Compare Held-Out Fold Macro F1**

Plot all five Development folds for the three classical models. Means and individual fold values should be considered together; these are not independent external replications.

In [ ]:
# ============ 09. Visualise Five-Fold Classical Model Performance ============

# -------------- 1. Plot Validation Fold Macro F1 by Classifier ---------------
fig, ax = plt.subplots(figsize=(10, 5))

for model_name, table in fold_results.groupby("model", sort=False):
    table = table.sort_values("fold")
    ax.plot(table["fold"], table["macro_f1"], marker="o",
            label=f"{model_name} (mean={table['macro_f1'].mean():.3f})")

ax.set(title="HARTH: 29-person development grouped CV",
       xlabel="Held-out fold", ylabel="Macro F1",
       ylim=(0, 1), xticks=[1, 2, 3, 4, 5])
ax.grid(alpha=0.25)
ax.legend()
fig.tight_layout()
save_figure(fig, output_dir / "03_development_cv_fold_macro_f1.png")
plt.show()

**10. Compare Random Forest and Logistic Regression by Participant**

Use paired fixed eight-class Macro F1 for the exact same held-out Development participants. Positive signed differences favour Random Forest; the comparison is descriptive and not a statistical significance test.

In [ ]:
# ============= 10. Compare Models on Identical Held-Out Subjects =============

# ------------------ 1. Match Participant-Level Model Scores ------------------
paired, paired_summary = compare_models_by_subject(
    subject_results,
    model_a="Random Forest",
    model_b="Logistic Regression",
    score_column="macro_f1_fixed_task",  # Use the same eight-class metric for both models.
)
display(paired.round(4))
display(pd.DataFrame([paired_summary]))

# --------------- 2. Visualise Signed Paired Score Differences ----------------
fig, ax = plt.subplots(figsize=(13, 5))

bar_colours = ["tab:blue" if value >= 0 else "tab:orange"
               for value in paired["difference_a_minus_b"]]
ax.bar(paired["subject_id"], paired["difference_a_minus_b"], color=bar_colours)
ax.axhline(0, color="black", linewidth=0.8)
ax.set(title="Per-subject Macro F1 difference: Random Forest minus Logistic Regression",
       xlabel="Held-out development participant", ylabel="Paired Macro F1 difference")
ax.tick_params(axis="x", rotation=70)
ax.grid(axis="y", alpha=0.2)
fig.tight_layout()
save_figure(fig, output_dir / "03_random_forest_logistic_regression_difference.png")
plt.show()

**11. Relate Model Performance to Participant Activity Support**

Merge eligible window counts, number of observed classes and fixed-task per-person model Macro F1. This comparison makes the limitations of small or unbalanced activity support visible.

In [ ]:
# =========== 11. Compare Individual Activity Support and Model F1 ============

# -------------- 1. Join Source Activity Counts and Model Scores --------------
person_support = pd.DataFrame({
    "subject_id": coverage.index,
    "eligible_windows": coverage.sum(axis=1).to_numpy(),
    "observed_classes": (coverage > 0).sum(axis=1).to_numpy(),
})
comparison = subject_results.pivot(
    index="subject_id", columns="model", values="macro_f1_fixed_task"
).reset_index()
participant_table = person_support.merge(comparison, on="subject_id", how="left")
display(participant_table.round(4))

print("Independent development subjects represented:", len(participant_table))

**12. Save Research Evidence and Package Current Outputs**

Write current source, class-coverage and evaluation outputs under the Stage 03 prefix. The full feature table is already saved in Section 04. Package only current Stage 03 results from the Colab output folder.

In [ ]:
# ============== 12. Save Research Results and Stage 03 Outputs ===============

# --------------- 1. Save Source and Fold Class-Coverage Tables ---------------
inventory.to_csv(output_dir / "03_development_source_inventory.csv",
                 index=False)
overview.to_csv(output_dir / "03_development_cohort_summary.csv",
                index=False)
excluded.to_csv(output_dir / "03_excluded_source_codes.csv",
                index=False)
coverage.rename_axis("subject_id").reset_index().to_csv(
    output_dir / "03_development_subject_class_coverage.csv", index=False
)
fold_audit.to_csv(output_dir / "03_development_fold_class_audit.csv",
                  index=False)
subject_support.rename_axis("activity").reset_index(
    name="supporting_subjects"
).to_csv(
    output_dir / "03_development_activity_subject_support.csv",
    index=False,
)

# ----------- 2. Save Cross-Validation and Paired Comparison Tables -----------
summary.to_csv(output_dir / "03_development_cv_model_summary.csv",
               index=False)
fold_results.to_csv(output_dir / "03_development_cv_fold_metrics.csv",
                    index=False)
subject_results.to_csv(
    output_dir / "03_development_cv_participant_metrics.csv", index=False
)
paired.to_csv(
    output_dir / "03_paired_model_participant_comparison.csv", index=False
)
pd.DataFrame([paired_summary]).to_csv(
    output_dir / "03_paired_model_summary.csv", index=False
)
participant_table.to_csv(
    output_dir / "03_development_participant_comparison.csv", index=False
)

# --------- 3. Display Results and Download the Current Stage Package ---------
display(summary.round(4))
display(pd.DataFrame([paired_summary]).round(4))
print("Development subjects:", metadata["subject_id"].nunique())
print("Eight-class windows:", len(metadata))
print("Original reserved Test scored: NO")
print("Saved research outputs:", output_dir)
download_outputs_from_colab(
    folder="wearable_activity_recognition/output", stage="03"
)

**13. Scientific Interpretation and Next Research Boundary**

The full-Development source audit and the five-fold classical model evaluation now consume the same aligned 29-person Development feature table. Ten of these participants' windows and features are read from Stage 02 outputs, and the remaining 19 recordings are processed once with unchanged source-safe windowing and feature methods.

**Interpretation:** All model evaluation is participant-disjoint. The fold-level fixed eight-class Macro F1 and the per-person fixed eight-class Macro F1 should not be treated as interchangeable summaries. The paired Random Forest-minus-Logistic Regression comparison uses the same held-out participants and is descriptive rather than a significance test. Activity support is reported both as eligible windows and as independent participants; overlapping windows are not independent observation units.

**Evaluation boundary:** Neither S023 nor S032 is loaded for Development model fitting or scoring here. Earlier descriptive examination of those reserved recordings means later same-study assessment cannot be called strictly blind or external validation. No final Test inference is performed.

**Next dependency:** Subsequent statistical-feature analyses can read `03_development29_statistical_features.csv` directly. A separate temporal CNN stage operates on original windows, not the 30-column statistical-feature representation, so its raw-signal preparation remains a different scientific operation.

**Execution status:** This consolidated notebook is new. Its results require current Colab execution; values from the earlier separate 08 and 09 notebooks must not be presented as verified results of this version.